# 0. les biblios

In [91]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Charger erp_business_partners.csv

In [92]:
# Charger les données
df = pd.read_csv("erp_business_partners.csv")

In [93]:
# 1. Dimensions et types de colonnes
print("Dimensions :", df.shape)
print("\nTypes de colonnes :")
df.info()

Dimensions : (80, 11)

Types de colonnes :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 11 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   partner_code                  80 non-null     object
 1   partner_name                  80 non-null     object
 2   partner_role                  80 non-null     object
 3   external_registration_number  66 non-null     object
 4   billing_region                79 non-null     object
 5   shipping_region               80 non-null     object
 6   default_currency              80 non-null     object
 7   payment_terms                 80 non-null     object
 8   partner_status                80 non-null     object
 9   risk_class                    80 non-null     object
 10  onboarding_date               80 non-null     object
dtypes: object(11)
memory usage: 7.0+ KB


In [94]:
# 2. Échantillon des données
print("\nPremières lignes :")
display(df.head())


Premières lignes :


,partner_code,partner_name,partner_role,external_registration_number,billing_region,shipping_region,default_currency,payment_terms,partner_status,risk_class,onboarding_date
0,BP-30001,Marula Cold Store Cooperative,supplier_customer,REG-AVL-20012,Avelora Federation,Eastern Avelora Agricultural Cluster,AVL,NET45,active,review,2001-02-01
1,BP-30002,Blue Heron Monitoring Cooperative,customer,REG-KNT-20037,Kintara Republic,Kintara Karst Cluster,KRN,NET45,active,standard,2012-03-01
2,BP-30003,Orra Technical Services,supplier,REG-AVL-20022,Avelora Federation,Eastern Avelora Agricultural Cluster,AVL,PUBLIC30,active,review,2008-04-01
3,BP-30004,Common Ground Climate Forum,service_partner,REG-VLN-01008,Valenne Republic,Valenne Central District,VLN,PREPAID,active,review,2014-05-01
4,BP-30005,Warden Medical Ventures,service_partner,REG-VLN-01016,Valenne Republic,Valenne Republic,VLN,PREPAID,inactive,standard,2013-06-01


In [95]:
# 3. Doublons
print("Nombre de lignes dupliquées (strictement identiques) :", df.duplicated().sum())
print("Nombre de partner_code dupliqués :", df['partner_code'].duplicated().sum())

# Afficher les doublons de partner_code s'il y en a
dupes = df[df['partner_code'].duplicated(keep=False)]
if not dupes.empty:
    display(dupes.sort_values('partner_code'))

Nombre de lignes dupliquées (strictement identiques) : 0
Nombre de partner_code dupliqués : 0


# 2. Valeurs manquantes

### a. Constat du problème

In [96]:
# 4. Valeurs manquantes
print("Valeurs manquantes par colonne :")
print(df.isna().sum())

print("\nPourcentage de manquants par colonne :")
print((df.isna().sum() / len(df) * 100).round(2))

Valeurs manquantes par colonne :
partner_code                     0
partner_name                     0
partner_role                     0
external_registration_number    14
billing_region                   1
shipping_region                  0
default_currency                 0
payment_terms                    0
partner_status                   0
risk_class                       0
onboarding_date                  0
dtype: int64

Pourcentage de manquants par colonne :
partner_code                     0.00
partner_name                     0.00
partner_role                     0.00
external_registration_number    17.50
billing_region                   1.25
shipping_region                  0.00
default_currency                 0.00
payment_terms                    0.00
partner_status                   0.00
risk_class                       0.00
onboarding_date                  0.00
dtype: float64


In [97]:
# Remplacer les valeurs manquantes par une mention explicite 'UNKNOWN' ou 'NON_DISPONIBLE'
df['external_registration_number'] = df['external_registration_number'].fillna('NON_DISPONIBLE')

# Vérification : il ne doit plus y avoir de valeurs manquantes
print("Valeurs manquantes restantes :", df['external_registration_number'].isna().sum())
print("Répartition après nettoyage :")
print(df['external_registration_number'].value_counts().head())

Valeurs manquantes restantes : 0
Répartition après nettoyage :
external_registration_number
NON_DISPONIBLE    14
REG-KNT-20037      1
REG-AVL-20012      1
REG-AVL-20022      1
REG-VLN-01008      1
Name: count, dtype: int64


In [98]:
# 1. Identifier la ligne avec la valeur manquante dans billing_region
missing_billing = df[df['billing_region'].isna()]
display(missing_billing)

# 2. Vérifier la correspondance entre shipping_region et billing_region pour les autres partenaires
print("Correspondances fréquentes entre shipping_region et billing_region :")
print(df[['shipping_region', 'billing_region']].drop_duplicates().head(10))

,partner_code,partner_name,partner_role,external_registration_number,billing_region,shipping_region,default_currency,payment_terms,partner_status,risk_class,onboarding_date
28,BP-30029,Kestrel Conservation Services,supplier,REG-KNT-20014,NaN,Kintara Karst Cluster,KRN,PREPAID,active,standard,2015-06-01


Correspondances fréquentes entre shipping_region et billing_region :
                         shipping_region        billing_region
0   Eastern Avelora Agricultural Cluster    Avelora Federation
1                  Kintara Karst Cluster      Kintara Republic
3               Valenne Central District      Valenne Republic
4                       Valenne Republic      Valenne Republic
6            Nordhaven Maritime District       Nordhaven Union
9                   Tembara Commonwealth  Tembara Commonwealth
11               Tembara Coastal Cluster  Tembara Commonwealth
12                       Nordhaven Union       Nordhaven Union
21                      Kintara Republic      Kintara Republic
28                 Kintara Karst Cluster                   NaN


In [99]:
# Imputer billing_region manquant en se basant sur shipping_region
df.loc[df['billing_region'].isna() & (df['shipping_region'] == 'Kintara Karst Cluster'), 'billing_region'] = 'Kintara Republic'

# Vérification : il ne doit plus y avoir de NaN dans billing_region
print("Valeurs manquantes dans billing_region après correction :", df['billing_region'].isna().sum())

Valeurs manquantes dans billing_region après correction : 0


# 3. Doublons

### a- Doublons de lignes

In [100]:
print("Nombre de lignes dupliquées :", df.duplicated().sum())
display(df[df.duplicated(keep=False)].sort_values('partner_code'))

Nombre de lignes dupliquées : 0


,partner_code,partner_name,partner_role,external_registration_number,billing_region,shipping_region,default_currency,payment_terms,partner_status,risk_class,onboarding_date


### b- Doublons de colonnes 

In [101]:
# 1. Vérifier si des noms de colonnes sont strictement identiques
noms_doublons = df.columns[df.columns.duplicated()]
print("Noms de colonnes en double :", noms_doublons.tolist())

# 2. Vérifier si le contenu de certaines colonnes est 100% identique
contenu_doublons = df.columns[df.T.duplicated()]
print("Colonnes avec un contenu dupliqué :", contenu_doublons.tolist())

Noms de colonnes en double : []
Colonnes avec un contenu dupliqué : []


In [102]:
# Vérifier le taux de correspondance exacte entre les deux colonnes
correspondance_exacte = (df['billing_region'] == df['shipping_region']).mean() * 100
print(f"Pourcentage de lignes où billing_region == shipping_region : {correspondance_exacte:.2f}%")

# Afficher une table de contingence croisée pour voir les liens logiques
print("\nTable de croisement (Billing vs Shipping) :")
display(pd.crosstab(df['billing_region'], df['shipping_region'], dropna=False))

Pourcentage de lignes où billing_region == shipping_region : 17.50%

Table de croisement (Billing vs Shipping) :


shipping_region,Avelora Federation,Eastern Avelora Agricultural Cluster,Kintara Karst Cluster,Kintara Republic,Nordhaven Maritime District,Nordhaven Union,Tembara Coastal Cluster,Tembara Commonwealth,Valenne Central District,Valenne Republic
billing_region,,,,,,,,,,
Avelora Federation,3,13,0,0,0,0,0,0,0,0
Kintara Republic,0,0,9,3,0,0,0,0,0,0
Nordhaven Union,0,0,0,0,9,3,0,0,0,0
Tembara Commonwealth,0,0,0,0,0,0,12,1,0,0
Valenne Republic,0,0,0,0,0,0,0,0,23,4


# 4. Formats incohérents

### Les types de chaque colonnes :

In [103]:
print(df.dtypes)

partner_code                    object
partner_name                    object
partner_role                    object
external_registration_number    object
billing_region                  object
shipping_region                 object
default_currency                object
payment_terms                   object
partner_status                  object
risk_class                      object
onboarding_date                 object
dtype: object


### - Vérifier le format de partner_code

In [104]:
pattern_partner = r'^BP-\d+$'
non_conformes_partner = df[~df['partner_code'].str.match(pattern_partner, na=False)]
print("partner_code non conformes :", len(non_conformes_partner))

partner_code non conformes : 0


### - Vérifier le format de partner_name

In [105]:
print(df['partner_name'].value_counts(dropna=False))

partner_name
Marula Cold Store Cooperative          1
Blue Heron Monitoring Cooperative      1
Orra Technical Services                1
Common Ground Climate Forum            1
Warden Medical Ventures                1
                                      ..
Avelora Council of State               1
Velin Packaging Industries             1
Meridian BioLogistics                  1
Kintara Field Equipment Cooperative    1
Garnet Scientific Instruments          1
Name: count, Length: 80, dtype: int64


### - Vérifier le format partner_role (catégorielles)

In [106]:
print(df['partner_role'].value_counts(dropna=False))

partner_role
customer             32
service_partner      19
supplier             13
carrier               9
supplier_customer     6
Supplier              1
Name: count, dtype: int64


### - Vérifier le format external_registration_number

In [107]:
pattern_reg = r'^(REG-[A-Z]+-\d+|NON_DISPONIBLE)$'
non_conformes_reg = df[~df['external_registration_number'].astype(str).str.match(pattern_reg, na=False)]
print(f"\n external_registration_number non conformes : {len(non_conformes_reg)} ")
if len(non_conformes_reg) > 0:
    display(non_conformes_reg[['partner_code', 'external_registration_number']])


 external_registration_number non conformes : 0 


### - Vérifier le format billing_region (catégorielles)

In [108]:
print(df['billing_region'].value_counts(dropna=False))

billing_region
Valenne Republic        27
Avelora Federation      16
Tembara Commonwealth    13
Kintara Republic        12
Nordhaven Union         12
Name: count, dtype: int64


### - Vérifier le format shipping_region (catégorielles)

In [109]:
print(df['shipping_region'].value_counts(dropna=False))

shipping_region
Valenne Central District                23
Eastern Avelora Agricultural Cluster    13
Tembara Coastal Cluster                 12
Kintara Karst Cluster                    9
Nordhaven Maritime District              9
Valenne Republic                         4
Kintara Republic                         3
Nordhaven Union                          3
Avelora Federation                       3
Tembara Commonwealth                     1
Name: count, dtype: int64


### - Vérifier le format default_currency

In [110]:
print(df['default_currency'].value_counts(dropna=False))

default_currency
VLN    27
AVL    16
TMB    13
KRN    12
NDH    12
Name: count, dtype: int64


### - Vérifier le format payment_terms (catégorielles)

In [111]:
print(df['payment_terms'].value_counts(dropna=False))

payment_terms
PUBLIC30    20
NET30       17
NET45       15
PREPAID     15
NET15       13
Name: count, dtype: int64


### - Vérifier le format partner_status (catégorielles ordonnées)

In [112]:
print(df['partner_status'].value_counts(dropna=False))

partner_status
active      72
inactive     8
Name: count, dtype: int64


### - Vérifier le format risk_class (catégorielles ordonnées)

In [113]:
print(df['risk_class'].value_counts(dropna=False))

risk_class
standard    37
review      19
low         15
elevated     9
Name: count, dtype: int64


### - Vérifier le format onboarding_date

In [114]:
pattern_date = r'^\d{4}-\d{2}-\d{2}$'
non_conformes_date = df[~df['onboarding_date'].astype(str).str.match(pattern_date, na=False)]
print("\nonboarding_date au format non conforme :", len(non_conformes_date))
# Afficher la ligne avec le format de date non conforme
display(non_conformes_date)


onboarding_date au format non conforme : 1


,partner_code,partner_name,partner_role,external_registration_number,billing_region,shipping_region,default_currency,payment_terms,partner_status,risk_class,onboarding_date
20,BP-30021,Asterion Clinical Institute,customer,REG-VLN-01002,Valenne Republic,Valenne Central District,VLN,PREPAID,active,review,01/10/2003


In [115]:
# Convertir en gérant les formats mixtes et le format européen (jour en premier)
df['onboarding_date'] = pd.to_datetime(df['onboarding_date'], format='mixed', dayfirst=True).dt.strftime('%Y-%m-%d')

# Vérification finale : il ne doit plus y avoir d'anomalie de format
non_conformes_date_fin = df[~df['onboarding_date'].astype(str).str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)]
print("Dates non conformes restantes :", len(non_conformes_date_fin))

Dates non conformes restantes : 0


# 5. Télécharger le csv nettoyé

In [116]:
df.to_csv('erp_business_partners_cleaned.csv', index=False)